In [94]:
import pandas as pd

import pickle
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import root_mean_squared_error

## Q1. Downloading the data

In [95]:
df = pd.read_parquet('./data/yellow_tripdata_2023-01.parquet')
print(df.shape[1])

19


## Q2. Computing duration

In [96]:
df.head(3)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.0,0.5,4.0,0.0,1.0,16.9,2.5,0.0
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.0,0.5,15.0,0.0,1.0,34.9,2.5,0.0


In [97]:
df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
print(df['duration'].std())

42.594351241920904


## Q3. Dropping outliers

In [98]:
df['duration'].describe()

count    3.066766e+06
mean     1.566900e+01
std      4.259435e+01
min     -2.920000e+01
25%      7.116667e+00
50%      1.151667e+01
75%      1.830000e+01
max      1.002918e+04
Name: duration, dtype: float64

In [99]:
rows_before = df.shape[0]

df = df[(df.duration >= 1) & (df.duration <= 60)]

rows_after = df.shape[0]

print(rows_before, rows_after)
print(f"Percentage of actual data: {100*rows_after/rows_before:.2f}%")


3066766 3009173
Percentage of actual data: 98.12%


## Q4. One-hot encoding

In [100]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

df[categorical] = df[categorical].astype(str)

In [101]:
train_dicts = df[categorical + numerical].to_dict(orient='records')

In [102]:
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

In [103]:
X_train.shape

(3009173, 516)

## Q5. Training a model

In [104]:
target = 'duration'
y_train = df[target].values

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)

root_mean_squared_error(y_train, y_pred)

7.658406414857231

## Q6. Evaluating the model

In [105]:
def read_dataframe(path):
    dataframe = pd.read_parquet(path)
    return dataframe

def normalized_data(dataframe, dv):

    dataframe['duration'] = dataframe.tpep_dropoff_datetime - dataframe.tpep_pickup_datetime
    dataframe.duration = dataframe.duration.apply(lambda td: td.total_seconds() / 60)

    dataframe = dataframe[(dataframe.duration >= 1) & (dataframe.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    numerical = ['trip_distance']

    dataframe[categorical] = dataframe[categorical].astype(str)

    dicts = dataframe[categorical + numerical].to_dict(orient='records')

    X = dv.transform(dicts)

    y = dataframe['duration'].values

    return X, y

In [106]:
df_val = read_dataframe('./data/yellow_tripdata_2023-02.parquet')
X_normalized, y_normalized = normalized_data(df_val,dv)

y_val = y_normalized
y_pred = lr.predict(X_normalized)

print(f"RMSE on validation: {root_mean_squared_error(y_val, y_pred)}")




/tmp/ipykernel_10550/1090217761.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe[categorical] = dataframe[categorical].astype(str)


RMSE on validation: 7.820106205185956
